In [ ]:
import os, sys
from pathlib import Path

# Locate the repo from the notebook's working directory (works anywhere).
for _c in (Path.cwd(), *Path.cwd().parents):
    if (_c / "assembloid_sync" / "__init__.py").exists():
        sys.path.insert(0, str(_c)); break
else:
    raise SystemExit("Run this notebook from inside the repo, or add it to sys.path manually")

# Point this at your dataset folder (or set ASSEMBLOID_SYNC_DATASET):
DATASET = os.environ.get("ASSEMBLOID_SYNC_DATASET", "")
if not DATASET:
    raise SystemExit("Set DATASET above, or the ASSEMBLOID_SYNC_DATASET environment variable")

from assembloid_sync import config, layout
ds = Path(DATASET)
cfg = config.load_config(ds)
print("dataset:", ds.name, "| frame rate:", config.resolve_frame_rate(ds, cfg))

In [ ]:
# --- run CNMF-E (identical to the batch stage) ---
import os
temp = layout.caiman_temp(ds); temp.mkdir(parents=True, exist_ok=True)
os.environ["CAIMAN_TEMP"] = str(temp)
from assembloid_sync import stage_denoise
cnm = stage_denoise.fit(ds, cfg)

In [ ]:
# --- inspect components (the old notebook's cell 3, interactive-only) ---
import matplotlib.pyplot as plt
import caiman as cm
# first 1000 frames is plenty for a correlation-image preview; if this recording
# is shorter than that, narrow the range below.
movie = cm.load(str(layout.raw_tif(ds)), subindices=range(0, 1000))
corr_img = movie.local_correlations(swap_dim=False)
if cnm.estimates.idx_components is not None and len(cnm.estimates.idx_components):
    cnm.estimates.plot_contours(img=corr_img, idx=cnm.estimates.idx_components)
plt.show()
cnm.estimates.view_components(img=corr_img)

In [ ]:
# --- save the denoised movie (chunked float32) ---
stage_denoise.write_denoised(cnm, layout.denoised_tif(ds), chunk=cfg["denoise"]["chunk_size"])
print("saved", layout.denoised_tif(ds))